In [ ]:
!pip install google-adk

In [ ]:
import time
from typing import Dict, Any

# Mocking user log callbacks for demonstration
def agent_interaction_logger(session_id: str, agent_name: str, event_type: str, data: Any):
    timestamp = time.strftime('%Y-%m-%d %H:%M:%S')
    print(f"[{timestamp}] [Session: {session_id}] Agent '{agent_name}' triggered {event_type}.")
    print(f"    -> Data: {data}\n" + "-"*50)

In [ ]:
from google.adk.tools import google_search # Native ADK open-source tool identifier

def mock_nws_weather_tool(city: str) -> str:
    """Fetches weather details from NWS for a specific location."""
    # Simulated NWS API response pattern
    city_lower = city.lower()
    if "york" in city_lower:
        return "NWS Report: New York is currently 72°F, Partly Cloudy with 10% precipitation chance."
    elif "francisco" in city_lower:
        return "NWS Report: San Francisco is 61°F, Breezy/Foggy morning opening into sun."
    else:
        return f"NWS Report: {city.title()} is 70°F with comfortable clear skies."

def mock_maps_route_tool(origin: str, destination: str) -> str:
    """Calculates route details using Google Maps matrix rules."""
    return f"Maps Route: Driving from {origin} to {destination} takes approximately 45 mins via Interstate main route. No major delays reported."

In [ ]:
from google.adk import Agent
from google.adk.agents import LlmAgent, SequentialAgent

# 1. Weather Forecast Specialist Agent
weather_agent = LlmAgent(
    name="weather_forecaster",
    model="gemini-flash-latest",
    instruction="You provide exact meteorological details using your provided NWS tool.",
    tools=[mock_nws_weather_tool]
)

# 2. Search & Info Agent
search_agent = LlmAgent(
    name="internet_researcher",
    model="gemini-flash-latest",
    instruction="Search the web for any context elements the user requests.",
    tools=[google_search]
)

# 3. Routing Agent
routing_agent = LlmAgent(
    name="maps_navigator",
    model="gemini-flash-latest",
    instruction="Generate clean driving/transit updates based on origin and destination parameters.",
    tools=[mock_maps_route_tool]
)

# 4. Sequenced Quality Assurance (Refiner) Agent
qa_refiner_agent = LlmAgent(
    name="qa_refiner",
    model="gemini-flash-latest",
    instruction="Review the cumulative logs generated by prior agents. Clean up formatting, resolve contradictions, and output a polished unified brief."
)

# Assemble into a sequential correction and refinement engine
refinement_pipeline = SequentialAgent(
    name="processing_pipeline",
    sub_agents=[weather_agent, search_agent, routing_agent, qa_refiner_agent]
)

In [ ]:
# Root Agent defining capability parameters
root_coordinator = LlmAgent(
    name="root_coordinator",
    model="gemini-flash-latest",
    instruction=(
        "You are the central CEO coordinator agent. You describe system capabilities to the user. "
        "When given a verified structural request, hand off processing entirely to the 'processing_pipeline'."
    ),
    sub_agents=[refinement_pipeline]
)

In [ ]:
def validate_and_execute(user_query: dict):
    # Validation Layer
    required_keys = ["origin", "destination", "topic"]
    for key in required_keys:
        if key not in user_query or not str(user_query[key]).strip():
            print(f"Validation Error: High priority parameter '{key}' missing or empty.")
            return None

    print("User input validation passed successfully!\n")

    # Session tracking setup
    session_id = "test_session_001"

    # Simulate execution callback tracking
    agent_interaction_logger(session_id, root_coordinator.name, "USER_REQUEST_RECEIVED", user_query)

    # In a full runtime, you initiate a Session runner:
    # runner = Runner(agent=root_coordinator, session=Session())
    # response = runner.run(f"Process this structured trip: {user_query}")

    # Simulating structural fallback pipeline response across the sub-agents for demo
    print(f"[{root_coordinator.name}] delegating to internal pipeline sub-agents...\n")

    w_res = mock_nws_weather_tool(user_query["destination"])
    agent_interaction_logger(session_id, weather_agent.name, "TOOL_EXECUTION", w_res)

    m_res = mock_maps_route_tool(user_query["origin"], user_query["destination"])
    agent_interaction_logger(session_id, routing_agent.name, "TOOL_EXECUTION", m_res)

    final_output = (
        f"### Final Validated Trip Brief Summary:\n"
        f"* **Weather**: {w_res}\n"
        f"* **Route**: {m_res}\n"
        f"* **Research Topic Notes**: Handled general context inquiries matching '{user_query['topic']}' successfully."
    )

    agent_interaction_logger(session_id, qa_refiner_agent.name, "PIPELINE_FINAL_REFINEMENT", "Success")
    return final_output

In [ ]:
import time
from typing import Dict, Any

# 1. Define the missing logging function explicitly
def agent_interaction_logger(session_id: str, agent_name: str, event_type: str, data: Any):
    timestamp = time.strftime('%Y-%m-%d %H:%M:%S')
    print(f"[{timestamp}] [Session: {session_id}] Agent '{agent_name}' triggered {event_type}.")
    print(f"    -> Data: {data}\n" + "-"*50)

# 2. Define your test validation runner
def validate_and_execute(user_query: dict):
    required_keys = ["origin", "destination", "topic"]
    for key in required_keys:
        if key not in user_query or not str(user_query[key]).strip():
            print(f"Validation Error: High priority parameter '{key}' missing or empty.")
            return None

    print("User input validation passed successfully!\n")

    session_id = "test_session_001"

    # This will now successfully find the function defined right above it!
    agent_interaction_logger(session_id, "root_coordinator", "USER_REQUEST_RECEIVED", user_query)

    print("[root_coordinator] delegating to internal pipeline sub-agents...\n")

    w_res = mock_nws_weather_tool(user_query["destination"])
    agent_interaction_logger(session_id, "weather_forecaster", "TOOL_EXECUTION", w_res)

    m_res = mock_maps_route_tool(user_query["origin"], user_query["destination"])
    agent_interaction_logger(session_id, "maps_navigator", "TOOL_EXECUTION", m_res)

    final_output = (
        f"### Final Validated Trip Brief Summary:\n"
        f"* **Weather**: {w_res}\n"
        f"* **Route**: {m_res}\n"
        f"* **Research Topic Notes**: Handled general context inquiries matching '{user_query['topic']}' successfully."
    )

    agent_interaction_logger(session_id, "qa_refiner", "PIPELINE_FINAL_REFINEMENT", "Success")
    return final_output

# 3. Execute the test
test_input = {
    "origin": "New York",
    "destination": "San Francisco",
    "topic": "Sightseeing landmarks across California"
}

output_result = validate_and_execute(test_input)
print(output_result)

User input validation passed successfully!

[2026-07-09 17:40:10] [Session: test_session_001] Agent 'root_coordinator' triggered USER_REQUEST_RECEIVED.
    -> Data: {'origin': 'New York', 'destination': 'San Francisco', 'topic': 'Sightseeing landmarks across California'}
--------------------------------------------------
[root_coordinator] delegating to internal pipeline sub-agents...

[2026-07-09 17:40:10] [Session: test_session_001] Agent 'weather_forecaster' triggered TOOL_EXECUTION.
    -> Data: NWS Report: San Francisco is 61°F, Breezy/Foggy morning opening into sun.
--------------------------------------------------
[2026-07-09 17:40:10] [Session: test_session_001] Agent 'maps_navigator' triggered TOOL_EXECUTION.
    -> Data: Maps Route: Driving from New York to San Francisco takes approximately 45 mins via Interstate main route. No major delays reported.
--------------------------------------------------
[2026-07-09 17:40:10] [Session: test_session_001] Agent 'qa_refiner' trigge

In [ ]:
import os

# Create the folder matching your CLI target argument
os.makedirs("my_engine", exist_ok=True)

# Write an empty package initializer file
with open("my_engine/__init__.py", "w") as f:
    pass
print("Directory 'my_engine' initialized successfully!")

Directory 'my_engine' initialized successfully!


In [ ]:
%%writefile my_engine/agent.py
import time
from typing import Dict, Any
from google.adk import Agent
from google.adk.agents import LlmAgent, SequentialAgent
from google.adk.tools import google_search

# --- System Callbacks ---
def agent_interaction_logger(session_id: str, agent_name: str, event_type: str, data: Any):
    timestamp = time.strftime('%Y-%m-%d %H:%M:%S')
    print(f"[{timestamp}] [Session: {session_id}] Agent '{agent_name}' triggered {event_type}.")
    print(f"    -> Data: {data}\n" + "-"*50)

# --- Custom Tools ---
def mock_nws_weather_tool(city: str) -> str:
    city_lower = city.lower()
    if "greensboro" in city_lower:
        return "NWS Report: Greensboro is currently 78°F, Clear skies with stable conditions."
    return f"NWS Report: {city.title()} is 70°F with comfortable clear skies."

def mock_maps_route_tool(origin: str, destination: str) -> str:
    return f"Maps Route: Driving from {origin} to {destination} takes approximately 45 mins via primary highway routes."

# --- Build Sub-Agents ---
weather_agent = LlmAgent(name="weather_forecaster", model="gemini-flash-latest", tools=[mock_nws_weather_tool])
search_agent = LlmAgent(name="internet_researcher", model="gemini-flash-latest", tools=[google_search])
routing_agent = LlmAgent(name="maps_navigator", model="gemini-flash-latest", tools=[mock_maps_route_tool])
qa_refiner_agent = LlmAgent(name="qa_refiner", model="gemini-flash-latest")

refinement_pipeline = SequentialAgent(
    name="processing_pipeline",
    sub_agents=[weather_agent, search_agent, routing_agent, qa_refiner_agent]
)

# --- Root Coordinator Agent ---
root_coordinator = LlmAgent(
    name="root_coordinator",
    model="gemini-flash-latest",
    instruction="Coordinate user requirements using the internal processing_pipeline.",
    sub_agents=[refinement_pipeline]
)

Writing my_engine/agent.py


In [ ]:
# 1. Define your Google Cloud configuration variables
PROJECT_ID = "qwiklabs-gcp-04-3d3bb8d72ee3"  # <-- Replace with your actual GCP Project ID
REGION = "us-central1"               # <-- Change region if needed

In [ ]:
%%writefile my_engine/requirements.txt
google-adk
google-genai

Overwriting my_engine/requirements.txt


In [ ]:
%%writefile my_engine/__init__.py
from . import agent

Overwriting my_engine/__init__.py


In [ ]:
%%writefile my_engine/agent.py
from google.adk import Agent, App
from google.adk.agents import LlmAgent, SequentialAgent

# 1. Weather Agent with embedded mock behavioral rules
weather_agent = LlmAgent(
    name="weather_forecaster",
    model="gemini-flash-latest",
    instruction="If the user asks about the weather, respond with: 'NWS Report: The destination is currently 72°F, Partly Cloudy with stable conditions.'"
)

# 2. Routing Agent with embedded mock behavioral rules
routing_agent = LlmAgent(
    name="maps_navigator",
    model="gemini-flash-latest",
    instruction="If the user asks for routes, respond with: 'Maps Route: Driving via the interstate main route takes approximately 45 mins.'"
)

# 3. Refiner Agent to clean and format output
qa_refiner_agent = LlmAgent(
    name="qa_refiner",
    model="gemini-flash-latest",
    instruction="Review inputs from prior agents, remove any contradictions, and present a clean final trip brief."
)

# Combine into a sequential pipeline
refinement_pipeline = SequentialAgent(
    name="processing_pipeline",
    sub_agents=[weather_agent, routing_agent, qa_refiner_agent]
)

# 4. Root Coordinator Agent
root_coordinator = LlmAgent(
    name="root_coordinator",
    model="gemini-flash-latest",
    instruction="You are the orchestrator. Validate that the request has an origin and destination, then delegate to the processing_pipeline.",
    sub_agents=[refinement_pipeline]
)

# 5. Define Deployed App Entrypoint
app = App(
    root_agent=root_coordinator,
    name="my_engine"
)

Overwriting my_engine/agent.py


In [ ]:
# 2. Authenticate your Google Cloud account if you haven't already
!gcloud auth login

# Deploy using the direct target location
#!adk deploy agent_engine --project=$PROJECT_ID --region=$REGION my_engine

!adk deploy agent_engine --project=$PROJECT_ID --region=$REGION my_engine


You are running on a Google Compute Engine virtual machine.
It is recommended that you use service accounts for authentication.

You can run:

  $ gcloud config set account `ACCOUNT`

to switch accounts if necessary.

Your credentials may be visible to others with access to this
virtual machine. Are you sure you want to authenticate with
your personal account?

Do you want to continue (Y/n)?  Y

Go to the following link in your browser, and complete the sign-in prompts:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=32555940559.apps.googleusercontent.com&redirect_uri=https%3A%2F%2Fsdk.cloud.google.com%2Fauthcode.html&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fappengine.admin+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcompute+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Faccounts.

In [ ]:
# Set your active project ID
!gcloud config set project qwiklabs-gcp-04-3d3bb8d72ee3

# Enable the Vertex AI API
!gcloud services enable aiplatform.googleapis.com

# Enable the Cloud Resource Manager API
!gcloud services enable cloudresourcemanager.googleapis.com

[environment: untagged] Read more to tag: g.co/cloud/project-env-tag.
Updated property [core/project].


In [ ]:
%%writefile my_engine/agent.py
from google.adk import Agent, App

# Base root coordinator configuration
root_coordinator = Agent(
    name="root_coordinator",
    model="gemini-flash-latest",
    instruction=(
        "You are a helpful trip coordinator. You validate that the user's input "
        "has an origin and destination, and respond with a structured summary."
    )
)

# Exposed app entry point matching the 'my_engine' directory name
app = App(
    root_agent=root_coordinator,
    name="my_engine"
)

Overwriting my_engine/agent.py


In [ ]:
%%writefile my_engine/__init__.py
from . import agent

Overwriting my_engine/__init__.py


In [ ]:
%%writefile my_engine/requirements.txt
google-adk

Overwriting my_engine/requirements.txt


In [ ]:
PROJECT_ID = "qwiklabs-gcp-04-3d3bb8d72ee3"  # Your active project ID
REGION = "us-central1"

!adk deploy agent_engine --project=$PROJECT_ID --region=$REGION my_engine

/usr/local/lib/python3.12/dist-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()
Staging all files in: /content/my_engine_tmp20260709_194630
Staging all files in: /content/my_engine_tmp20260709_194630
Copying agent source code...
Copying agent source code complete.
Resolving files and dependencies...
Initializing Vertex AI...
Vertex AI initialized.
Created my_engine_tmp20260709_194630/agent_engine_app.py
Files and dependencies resolved
Deploying to agent engine...
Cleaning up the temp folder: my_engine_tmp20260709_194630
Deploy failed: Failed to create Agent Engine: {'code': 3, 'message': 'Reasoning Engine resource [projects/153302561600/locations/us-central1/reasoningEngines/1946784842779721728] failed to start and cannot serve traffic. Please refer to our troubleshooting pages (e.g., https://docs.cloud.google.com/gemini-enterprise-agent-platform/troubleshooting/agent-deployment) 

In [ ]:
# Set your active project ID
!gcloud config set project qwiklabs-gcp-04-3d3bb8d72ee3

# Enable the core foundational APIs required by ADK deployments
!gcloud services enable agentplatform.googleapis.com
!gcloud services enable aiplatform.googleapis.com
!gcloud services enable cloudresourcemanager.googleapis.com

[environment: untagged] Read more to tag: g.co/cloud/project-env-tag.
Updated property [core/project].
ERROR: (gcloud.services.enable) PERMISSION_DENIED: Not found or permission denied for service(s): agentplatform.googleapis.com.
Help Token: AdZh9GfrF48D7ekTzGUwfT9gfZ69CkMbqVKFFQA3DxLI_YQUiK_j6FfiTwwfPGKiraID2PiLucBvm0nCLh5TsmDlU5XuP12ybrXsWtZx2nz7pV6d. This command is authenticated as student-00-86fa165fa519@qwiklabs.net which is the active account specified by the [core/account] property
- '@type': type.googleapis.com/google.rpc.PreconditionFailure
  violations:
  - subject: ?error_code=220002&services=agentplatform.googleapis.com
    type: googleapis.com
- '@type': type.googleapis.com/google.rpc.ErrorInfo
  domain: serviceusage.googleapis.com
  metadata:
    services: agentplatform.googleapis.com
  reason: SERVICE_CONFIG_NOT_FOUND_OR_PERMISSION_DENIED


In [ ]:
# Set your project ID
!gcloud config set project qwiklabs-gcp-04-3d3bb8d72ee3

# Enable the universal Vertex AI service wrapper
!gcloud services enable aiplatform.googleapis.com

[environment: untagged] Read more to tag: g.co/cloud/project-env-tag.
Updated property [core/project].


In [ ]:
PROJECT_ID = "qwiklabs-gcp-04-3d3bb8d72ee3"  # Replace with your active project ID
REGION = "us-central1"

!adk deploy agent_engine --project=$PROJECT_ID --region=$REGION my_engine

/usr/local/lib/python3.12/dist-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()
Staging all files in: /content/my_engine_tmp20260709_200548
Staging all files in: /content/my_engine_tmp20260709_200548
Copying agent source code...
Copying agent source code complete.
Resolving files and dependencies...
Initializing Vertex AI...
Vertex AI initialized.
Created my_engine_tmp20260709_200548/agent_engine_app.py
Files and dependencies resolved
Deploying to agent engine...
Cleaning up the temp folder: my_engine_tmp20260709_200548
Deploy failed: Failed to create Agent Engine: {'code': 3, 'message': 'Reasoning Engine resource [projects/153302561600/locations/us-central1/reasoningEngines/2178720223589302272] failed to start and cannot serve traffic. Please refer to our troubleshooting pages (e.g., https://docs.cloud.google.com/gemini-enterprise-agent-platform/troubleshooting/agent-deployment) 

In [ ]:
%%writefile my_engine/agent.py
from google.adk import Agent
from google.adk.agents import LlmAgent, SequentialAgent

class MyMultiAgentEngine:
    """Class-based multi-agent interface optimized for Vertex AI Reasoning Engine serialization."""

    def __init__(self):
        # We define agents inside __init__ to prevent server-side global pickling conflicts
        self.weather_agent = LlmAgent(
            name="weather_forecaster",
            model="gemini-2.5-flash",
            instruction="Provide weather notes. Mock: 'NWS Report: 72°F, Partly Cloudy.'"
        )
        self.routing_agent = LlmAgent(
            name="maps_navigator",
            model="gemini-2.5-flash",
            instruction="Provide routing notes. Mock: 'Maps Route: Clear route, 45 mins.'"
        )
        self.qa_refiner_agent = LlmAgent(
            name="qa_refiner",
            model="gemini-2.5-flash",
            instruction="Merge details into a clean final response brief."
        )

        # Sequential pipeline
        self.pipeline = SequentialAgent(
            name="processing_pipeline",
            sub_agents=[self.weather_agent, self.routing_agent, self.qa_refiner_agent]
        )

        # Root Coordinator
        self.root_coordinator = LlmAgent(
            name="root_coordinator",
            model="gemini-2.5-flash",
            instruction="Validate inputs (must have origin/destination) and forward to processing_pipeline.",
            sub_agents=[self.pipeline]
        )

    def set_up(self) -> None:
        """Required lifecycle method for Vertex AI Reasoning Engine initialization."""
        pass

    def query(self, input_text: str) -> str:
        """Required execution entry point for Vertex AI Reasoning Engine requests."""
        # Simple local runtime validation fallback if executed directly
        if "origin" not in input_text.lower() or "destination" not in input_text.lower():
            return "Validation Error: Inputs must contain an origin and a destination."

        # In a full cloud environment, you trigger your runner execution framework
        return f"Engine processed input text successfully via coordinated pipelines: {input_text}"

Overwriting my_engine/agent.py


In [ ]:
%%writefile my_engine/__init__.py
from .agent import MyMultiAgentEngine

# This variable explicitly lets the loader know what to initialize
app = MyMultiAgentEngine()

Overwriting my_engine/__init__.py


In [ ]:
PROJECT_ID = "qwiklabs-gcp-04-3d3bb8d72ee3"  # Replace with your active project ID
REGION = "us-east1"

!adk deploy agent_engine --project=$PROJECT_ID --region=$REGION my_engine

/usr/local/lib/python3.12/dist-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()
Staging all files in: /content/my_engine_tmp20260709_203443
Staging all files in: /content/my_engine_tmp20260709_203443
Copying agent source code...
Copying agent source code complete.
Resolving files and dependencies...
Initializing Vertex AI...
Vertex AI initialized.
Created my_engine_tmp20260709_203443/agent_engine_app.py
Files and dependencies resolved
Deploying to agent engine...
Cleaning up the temp folder: my_engine_tmp20260709_203443
Deploy failed: Failed to create Agent Engine: {'code': 3, 'message': 'Reasoning Engine resource [projects/153302561600/locations/us-east1/reasoningEngines/3837031698147573760] failed to start and cannot serve traffic. Please refer to our troubleshooting pages (e.g., https://docs.cloud.google.com/gemini-enterprise-agent-platform/troubleshooting/agent-deployment) to 